In [1]:
import pyspark
from pyspark.sql import SparkSession

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA, Imputer
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.sql.functions import mean, col, expr
import numpy as np
import time

In [2]:
spark = SparkSession.builder.appName("ce53") \
    .master("local[*]") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.maxResultSize", "3g") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "25g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "16") \
    .config("spark.shuffle.partitions", "180") \
    .config("spark.kryoserializer.buffer.max", "256m") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.sql.execution.arrow.enabled", "true") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/07/05 14:31:28 WARN Utils: Your hostname, colin-MS-7977 resolves to a loopback address: 127.0.1.1; using 192.168.0.164 instead (on interface wlx3c52a1d3ccda)
24/07/05 14:31:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/05 14:31:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#spark.sparkContext.stop()

In [4]:
parquet_files = ["Parquet/part-00000-23fdcfa3-9dd3-4c72-886c-e945bfcf92e1-c000.snappy.parquet",
                 "Parquet/part-00000-2b76f9cc-0710-45e4-9e33-98ad5808ee79-c000.snappy.parquet",
                 "Parquet/part-00000-745e350a-da9e-4619-bd52-8cc23bb41ad5-c000.snappy.parquet",
                 "Parquet/part-00000-94d13437-ae00-4a8c-9f38-edd0196cfdee-c000.snappy.parquet",
                 "Parquet/part-00000-9a46dd05-4b06-4a39-a45b-5c8460b6c37b-c000.snappy.parquet",
                 "Parquet/part-00000-9ac876be-c07d-4a18-878d-959efa26f484-c000.snappy.parquet",
                 "Parquet/part-00000-9aeb279c-81c6-4481-9b30-d35d4d194fea-c000.snappy.parquet",
                 "Parquet/part-00000-b2b625bc-5816-4586-b977-35f9ed4487fd-c000.snappy.parquet",
                 "Parquet/part-00000-be6d0798-554d-4c7a-9fef-d4c07aa0ce19-c000.snappy.parquet",
                 "Parquet/part-00000-d28b031b-bff1-4e16-853a-9b7d896627e7-c000.snappy.parquet",
                 "Parquet/part-00000-d512890f-d1e9-49d5-a136-f87f0183cb4d-c000.snappy.parquet",
                 "Parquet/part-00000-ea53b0e8-d346-44e3-9a87-1f60ac35c610-c000.snappy.parquet",
                 "Parquet/part-00000-f9afaec0-242e-41e7-906d-a42681515d75-c000.snappy.parquet"]

In [5]:
#Read the parquet files
df = spark.read.parquet(*parquet_files, inferSchema=True)

24/07/05 14:31:30 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:31:30 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:31:30 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


In [6]:
#Get unique label counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

label_counts.show()

+--------------------+------+
|        label_tactic| count|
+--------------------+------+
|          Collection|     1|
| Command and Control|    17|
|   Credential Access|     1|
|     Defense Evasion|  3064|
|           Discovery| 16819|
|           Execution|    30|
|      Initial Access|    19|
|    Lateral Movement|    11|
|         Persistence|    10|
|Privilege Escalation|  3066|
|      Reconnaissance| 51492|
|Resource Development|275471|
|                none|350339|
+--------------------+------+



In [7]:
start_time = time.time()

#Drop labels and get remaining counts
labels_to_drop = ['Exfiltration',
                  'Initial Access',
                  'Lateral Movement', 
                  'Persistence',
                  'Credential Access', 
                  'Collection',
                  'Command and Control',
                  'Execution']
#                 'Defense Evasion'
#                 'Discovery']
#                 'Reconnaissance',
#                 'Privilege Escalation', 
#                 'Resource Development', 

df = df.filter(~col("label_tactic").isin(labels_to_drop))
#filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

#filtered_label_counts.show()

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.0452265739440918 seconds


In [8]:
#Drop uid feature
df = df.drop('uid')
df = df.drop('label_technique')
df = df.drop('label_binary')

In [9]:
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

filtered_label_counts.show()

+--------------------+------+
|        label_tactic| count|
+--------------------+------+
|     Defense Evasion|  3064|
|           Discovery| 16819|
|Privilege Escalation|  3066|
|      Reconnaissance| 51492|
|Resource Development|275471|
|                none|350339|
+--------------------+------+



In [10]:
df.printSchema()

root
 |-- community_id: string (nullable = true)
 |-- conn_state: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- history: string (nullable = true)
 |-- src_ip_zeek: string (nullable = true)
 |-- src_port_zeek: long (nullable = true)
 |-- dest_ip_zeek: string (nullable = true)
 |-- dest_port_zeek: long (nullable = true)
 |-- local_orig: boolean (nullable = true)
 |-- local_resp: boolean (nullable = true)
 |-- missed_bytes: long (nullable = true)
 |-- orig_bytes: long (nullable = true)
 |-- orig_ip_bytes: long (nullable = true)
 |-- orig_pkts: long (nullable = true)
 |-- proto: string (nullable = true)
 |-- resp_bytes: long (nullable = true)
 |-- resp_ip_bytes: long (nullable = true)
 |-- resp_pkts: long (nullable = true)
 |-- service: string (nullable = true)
 |-- ts: double (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- label_tactic: string (nullable = true)



In [11]:
start_time = time.time()

#Columns to index
columns_to_index = ['service', 
                    'conn_state', 
                    'history', 
                    'proto', 
                    'dest_ip_zeek', 
                    'community_id', 
                    'src_ip_zeek',
                    'datetime',
                    'local_resp',
                    'local_orig',
                    'label_tactic']

#Cast datetime, local_resp, local_orig to String
df = df.withColumn("datetime", col("datetime").cast("string"))
df = df.withColumn("local_resp", col("local_resp").cast("string"))
df = df.withColumn("local_orig", col("local_orig").cast("string"))

#Impute null values with empty string
for column in columns_to_index:
    df = df.fillna('', subset=[column])

In [12]:
#Split the into training and test sets
start_time = time.time()
train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.026039600372314453 seconds


In [13]:
#StringIndexer
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").setHandleInvalid("keep") for column in columns_to_index]

#Chain indexers together
pipeline = Pipeline(stages=indexers).fit(train_data)

#Fit and transform the data
train_data_indexed = pipeline.transform(train_data)
test_data_indexed = pipeline.transform(test_data)

#Drop original columns
train_data_indexed = train_data_indexed.drop(*columns_to_index)
train_data_indexed = train_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")
test_data_indexed = test_data_indexed.drop(*columns_to_index)
test_data_indexed = test_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")

#print("train_indexed columns: ", train_data_indexed.columns)
#print("test_indexed columns: ", test_data_indexed.columns)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 20.026707887649536 seconds


In [14]:
start_time = time.time()

#List of numeric column names
numeric_columns = ['resp_pkts', 
                   'orig_ip_bytes', 
                   'missed_bytes', 
                   'duration', 
                   'orig_pkts',
                   'resp_ip_bytes', 
                   'dest_port_zeek', 
                   'orig_bytes', 
                   'resp_bytes',
                   'src_port_zeek', 
                   'ts']

#Create Imputer
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

#Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data_indexed)

#Apply the Imputer to the training data
train_data_imputed = imputer_model.transform(train_data_indexed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Apply the Imputer to the test data
start_time = time.time()
test_data_imputed = imputer_model.transform(test_data_indexed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

train_data_imputed = train_data_imputed.drop(*numeric_columns)
test_data_imputed = test_data_imputed.drop(*numeric_columns)

#print("\nTrain data imputed: ", train_data_imputed.columns)
#print("\n")
#print("Test data imputed: ", test_data_imputed.columns)

Execution time: 1.592191219329834 seconds
Execution time: 0.018477201461791992 seconds


In [15]:
start_time = time.time()

#Create VectorAssembler
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed") or column.endswith("_indexed")]
#print("Columns to assemble: ", columns_to_assemble)

assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

#Transform the training data
train_data_assembled = assembler.transform(train_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Transform the test data
start_time = time.time()
test_data_assembled = assembler.transform(test_data_imputed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Select the features and label columns
train_data_assembled = train_data_assembled.select("features", "label_tactic")
test_data_assembled = test_data_assembled.select("features", "label_tactic")

#print("Train_data_assembled columns: ", train_data_assembled.columns)
#print("Test_data_assembled columns: ", test_data_assembled.columns)

Execution time: 0.7590901851654053 seconds
Execution time: 0.2929410934448242 seconds


In [16]:
from pyspark.ml.feature import StandardScaler

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic")

24/07/05 14:31:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/07/05 14:32:00 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/05 14:32:05 WARN DAGScheduler: Broadcasting large task binary with size 33.2 MiB


In [17]:
# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic")

In [18]:
# Define the PCA model
pca = PCA(k=3, inputCol="features_normalized", outputCol="pca_features")

# Fit the PCA model on the normalized training set
start_time = time.time()
pca_model = pca.fit(train_data_normalized)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 14:32:08 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/05 14:32:08 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:32:10 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/05 14:32:10 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:32:12 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/05 14:32:13 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:32:13 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been depre

Execution time: 19.748777389526367 seconds


24/07/05 14:32:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [19]:
# Apply PCA transformation to the training and test sets
train_pca = pca_model.transform(train_data_normalized)
test_pca = pca_model.transform(test_data_normalized)

In [20]:
# Drop the normalized column and rename the pca_features column
train_pca = train_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")
test_pca = test_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")

# Verify the changes
#train_pca.show()
#test_pca.show()

In [21]:
#Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)
ovr = OneVsRest(classifier=svm, labelCol="label_tactic")
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.014339447021484375 seconds


In [22]:
#Fit the model
start_time = time.time()
svm_model = ovr.fit(train_pca)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 14:32:27 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:32:28 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/05 14:32:33 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/05 14:32:34 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:32:34 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:32:34 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.py

Execution time: 846.7429416179657 seconds


In [23]:
#Make predictions
start_time = time.time()

predictions = svm_model.transform(test_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 14:46:32 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


Execution time: 0.7215480804443359 seconds


In [24]:
# Evaluate the model
# Calculate accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)

# Calculate precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Calculate recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Calculate F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

#Calculate FPR
evaluator_fprL = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="falsePositiveRateByLabel")
fprL_score = evaluator_fprL.evaluate(predictions)

#Calculate Weighted FPR
evaluator_fpr = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedFalsePositiveRate")
fpr_score = evaluator_fpr.evaluate(predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)
print("FPR by Label:", fprL_score)
print("Weighted FPR:", fpr_score)

24/07/05 14:46:37 WARN DAGScheduler: Broadcasting large task binary with size 33.4 MiB
24/07/05 14:46:38 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:46:38 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:46:38 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:46:38 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:46:38 WARN SQLConf: The SQL config 'spark.sql

Accuracy: 0.9046949877575907
Precision: 0.8431999000023916
Recall: 0.9046949877575907
F1-Score: 0.8704157828964266
FPR by Label: 0.13795896585570389
Weighted FPR: 0.07360951303251437


In [25]:
start_time = time.time()

#Extract predictions and labels
predictions_and_labels = predictions.select("prediction", "label_tactic")

#Calculate false positives and true negatives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic== 0)).count()
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic == 0)).count()

#Calculate FPR
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 14:47:59 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/05 14:48:09 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB


False Positive Rate: 0.004227695760695006
Execution time: 18.94048023223877 seconds


In [26]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import Row

# Convert DataFrame to RDD of tuples (prediction, label)
prediction_and_labels = predictions.select("prediction", "label_tactic") \
    .rdd.map(lambda row: (float(row['prediction']), float(row['label_tactic'])))


# Instantiate BinaryClassificationMetrics
metrics = BinaryClassificationMetrics(prediction_and_labels)

# Compute AUROC
auROC = metrics.areaUnderROC

# Print AUROC
print("Area under ROC = ", auROC)

/home/colin/.local/lib/python3.10/site-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(
24/07/05 14:48:18 WARN DAGScheduler: Broadcasting large task binary with size 33.4 MiB
24/07/05 14:48:18 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:48:20 WARN DAGScheduler: Broadcasting large task binary with size 33.4 MiB
24/07/05 14:48:21 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 14:48:21 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/0

Area under ROC =  0.9154489355848873


In [27]:
spark.sparkContext.stop()